In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, f1_score

# Importation spécifique pour le déséquilibre
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
!pip install imbalanced-learn


   ---------------------------------------- 0/2 [sklearn-compat]
   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# ==========================================
# 0. CHARGEMENT DES DONNÉES (À adapter)
# ==========================================
# Supposons que votre DataFrame s'appelle 'df'
df = pd.read_excel("D:/Projets/Projet_Bank_Taux_de_reconvertion/thera_bank.xlsx", sheet_name="Data", 
                   header=0, index_col=0)

In [25]:
# ==========================================
# 0.c CONVERSION DES COLONNES OUI/NON EN 0/1
# ==========================================

# Colonnes qui contiennent 'Oui'/'Non' et doivent devenir 0/1
colonnes_ouinon = ['Compte_Titre', 'Certificat_Depot', 'Services_Ligne', 'CreditCard']

mapping_ouinon = {
    'oui': 1, 'yes': 1, 'true': 1, 'vrai': 1, '1': 1,
    'non': 0, 'no': 0, 'false': 0, 'faux': 0, '0': 0
}

for col in colonnes_ouinon:
    if col in df.columns:
        print(f"🔧 Conversion de '{col}' ({df[col].dtype}) : {df[col].unique()}")
        df[col] = (df[col].astype(str).str.strip().str.lower()
                   .map(mapping_ouinon)
                   .fillna(0)
                   .astype(int))
        print(f"   → Résultat : {df[col].unique()}")

# Conversion de ZIP Code en string (car c'est une variable NOMINALE, pas un nombre)
if 'ZIP Code' in df.columns:
    print(f"\n🔧 Conversion de 'ZIP Code' de {df['ZIP Code'].dtype} vers str (variable nominale)")
    df['ZIP Code'] = df['ZIP Code'].astype(str)

# Conversion de A_Pret_Hypothecaire et Pret_Hypothecaire (sécurité)
for col in ['A_Pret_Hypothecaire', 'Prêt_Hypothecaire']:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = (df[col].astype(str).str.strip().str.lower()
                   .map(mapping_ouinon).fillna(0).astype(int))

print("\n✅ Conversion terminée.")
print(df[colonnes_ouinon + ['ZIP Code', 'A_Pret_Hypothecaire', 'Prêt_Hypothecaire']].dtypes)

🔧 Conversion de 'Compte_Titre' (str) : <ArrowStringArray>
['Oui', 'Non']
Length: 2, dtype: str
   → Résultat : [1 0]
🔧 Conversion de 'Certificat_Depot' (str) : <ArrowStringArray>
['Non', 'Oui']
Length: 2, dtype: str
   → Résultat : [0 1]
🔧 Conversion de 'Services_Ligne' (str) : <ArrowStringArray>
['Non', 'Oui']
Length: 2, dtype: str
   → Résultat : [0 1]
🔧 Conversion de 'CreditCard' (str) : <ArrowStringArray>
['Non', 'Oui']
Length: 2, dtype: str
   → Résultat : [0 1]

🔧 Conversion de 'ZIP Code' de int64 vers str (variable nominale)

✅ Conversion terminée.
Compte_Titre           int64
Certificat_Depot       int64
Services_Ligne         int64
CreditCard             int64
ZIP Code                 str
A_Pret_Hypothecaire    int64
Prêt_Hypothecaire      int64
dtype: object


In [27]:
# ==========================================
# 1. DÉFINITION DYNAMIQUE DES COLONNES
# ==========================================
num_cols_souhaitees = ['Age', 'Experience', 'Revenu', 'Depenses_Mensuelle', 'Family', 'Prêt_Hypothecaire']
ord_cols_souhaitees = ['Education', 'Categorie_Revenu', 'Categorie_Age']
nom_cols_souhaitees = ['ZIP Code']
bin_cols_souhaitees = ['A_Pret_Hypothecaire', 'Compte_Titre', 'Certificat_Depot', 
                       'Services_Ligne', 'CreditCard']

# Filtrage dynamique (on garde celles qui existent)
toutes_colonnes = set(df.columns) - {'Accepte_Pret'}
num_cols = [c for c in num_cols_souhaitees if c in toutes_colonnes]
ord_cols = [c for c in ord_cols_souhaitees if c in toutes_colonnes]
nom_cols = [c for c in nom_cols_souhaitees if c in toutes_colonnes]
bin_cols = [c for c in bin_cols_souhaitees if c in toutes_colonnes]

print(f"Num : {num_cols}")
print(f"Ord : {ord_cols}")
print(f"Nom : {nom_cols}")
print(f"Bin : {bin_cols}")

Num : ['Age', 'Experience', 'Revenu', 'Depenses_Mensuelle', 'Family', 'Prêt_Hypothecaire']
Ord : ['Education', 'Categorie_Revenu', 'Categorie_Age']
Nom : ['ZIP Code']
Bin : ['A_Pret_Hypothecaire', 'Compte_Titre', 'Certificat_Depot', 'Services_Ligne', 'CreditCard']


In [16]:
# Création de Categorie_Revenu
# Adaptez les seuils (bins) selon votre logique métier
df['Categorie_Revenu'] = pd.cut(
    df['Revenu'], 
    bins=[-1, 20000, 60000, float('inf')], 
    labels=['Faible', 'Moyen', 'Très élevé']
)

# Création de Categorie_Age
df['Categorie_Age'] = pd.cut(
    df['Age'], 
    bins=[-1, 30, 45, float('inf')], 
    labels=['Jeune', 'Adulte', 'Agé']
)

# --- A_Pret_Hypothecaire (version binaire) ---
# Votre logique : 0 si pas de prêt, 1 si a un prêt
if 'A_Pret_Hypothecaire' not in df.columns:
    if 'Prêt_Hypothecaire' in df.columns:
        print("⚠️ Création de 'A_Pret_Hypothecaire' (binaire) à partir de 'Prêt_Hypothecaire'...")
        df['A_Pret_Hypothecaire'] = (df['Prêt_Hypothecaire'] > 0).astype(int)
        print(f"   Distribution : {df['A_Pret_Hypothecaire'].value_counts().to_dict()}")
    else:
        print("❌ ERREUR : 'Prêt_Hypothecaire' manquant, impossible de créer 'A_Pret_Hypothecaire'")


⚠️ Création de 'A_Pret_Hypothecaire' (binaire) à partir de 'Prêt_Hypothecaire'...
   Distribution : {0: 3462, 1: 1538}


In [17]:
# Séparation des features (X) et de la cible (y)
X = df.drop(columns=['Accepte_Pret'])
y = df['Accepte_Pret']

# ==========================================
# 1. DÉFINITION DES COLONNES ET PRÉPROCESSEUR
# ==========================================
num_cols = ['Age', 'Revenu', 'Depenses_Mensuelle', 'Family']
ord_cols = ['Education', 'Categorie_Revenu', 'Categorie_Age']
nom_cols = ['ZIP Code']
bin_cols = ['Prêt_Hypothecaire', 'Compte_Titre', 'Certificat_Depot', 'Services_Ligne', 'CreditCard', 'A_Pret_Hypothecaire']

# Pipeline pour les variables numériques
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline pour les variables ordinales (Attention à l'ordre exact de vos catégories)
ord_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[
        ['Licence', 'Master'], # À ajuster selon vos données réelles
        ['Faible', 'Moyen', 'Très élevé'], 
        ['Jeune', 'Adulte']
    ]))
])

# Pipeline pour les variables nominales (ZIP Code)
nom_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Assemblage du préprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('ord', ord_transformer, ord_cols),
        ('nom', nom_transformer, nom_cols),
        ('bin', 'passthrough', bin_cols) 
    ])

In [18]:
# ==========================================
# 2. TRAIN_TEST_SPLIT STRATIFIÉ
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Taille du Train : {X_train.shape}, Taille du Test : {X_test.shape}")
print(f"Distribution de la cible dans le Train :\n{y_train.value_counts(normalize=True)}")



Taille du Train : (4000, 15), Taille du Test : (1000, 15)
Distribution de la cible dans le Train :
Accepte_Pret
Non    0.904
Oui    0.096
Name: proportion, dtype: float64


In [22]:
# ==========================================
# 2. CONSTRUCTION DYNAMIQUE DU PRÉPROCESSEUR
# ==========================================
transformers = []

# --- Numériques ---
if num_cols:
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    transformers.append(('num', num_transformer, num_cols))

# --- Ordinales : DÉTECTION 100% AUTOMATIQUE ---
if ord_cols:
    # Ordre métier souhaité (facultatif). Le code ne gardera QUE ce qui existe.
    ordres_metier = {
        'Education': ['Primaire', 'BEPC', 'Baccalauréat', 'Formation professionnelle',
                      'Licence', 'Master', 'Doctorat'],
        'Categorie_Revenu': ['Faible', 'Moyen', 'Très élevé'],
        'Categorie_Age': ['Enfant', 'Jeune', 'Adulte', 'Senior', 'Vieux']
    }
    
    categories_list = []
    for col in ord_cols:
        # 1. Récupérer les valeurs RÉELLES présentes dans le df
        valeurs_reelles = df[col].astype(str).dropna().unique().tolist()
        print(f"\n📊 '{col}' contient : {sorted(valeurs_reelles)}")
        
        # 2. Si un ordre métier est défini, filtrer pour ne garder que les valeurs existantes
        if col in ordres_metier:
            ordre_voulu = ordres_metier[col]
            # Ne garder que les catégories présentes dans df, dans l'ordre métier
            ordonnees = [v for v in ordre_voulu if v in valeurs_reelles]
            # Ajouter celles qui étaient imprévues (à la fin, pour ne rien casser)
            imprevues = [v for v in valeurs_reelles if v not in ordonnees]
            if imprevues:
                print(f"   ⚠️ Catégories imprévues ajoutées en fin : {imprevues}")
            ordonnees.extend(imprevues)
        else:
            # Sinon, tri alphabétique (au pire)
            ordonnees = sorted(valeurs_reelles)
        
        categories_list.append(ordonnees)
        print(f"   ✅ Ordre final : {ordonnees}")
    
    ord_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(
            categories=categories_list,
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ))
    ])
    transformers.append(('ord', ord_transformer, ord_cols))

# --- Nominales ---
if nom_cols:
    nom_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])
    transformers.append(('nom', nom_transformer, nom_cols))

# --- Binaires ---
if bin_cols:
    transformers.append(('bin', 'passthrough', bin_cols))

# Assemblage final
preprocessor = ColumnTransformer(transformers=transformers)


📊 'Education' contient : ['Formation professionnelle', 'Licence', 'Master']
   ✅ Ordre final : ['Formation professionnelle', 'Licence', 'Master']

📊 'Categorie_Revenu' contient : ['Faible', 'Moyen', 'Très élevé']
   ✅ Ordre final : ['Faible', 'Moyen', 'Très élevé']

📊 'Categorie_Age' contient : ['Adulte', 'Agé', 'Jeune']
   ⚠️ Catégories imprévues ajoutées en fin : ['Agé']
   ✅ Ordre final : ['Jeune', 'Adulte', 'Agé']


In [24]:
print("🔍 Diagnostic des types de colonnes :\n")
for col in X_train.columns:
    dtype = X_train[col].dtype
    exemple = X_train[col].dropna().iloc[0] if len(X_train[col].dropna()) > 0 else "VIDE"
    type_valeur = type(exemple).__name__
    flag = "❌ PROBLÈME" if dtype == 'object' and col not in ord_cols + nom_cols else "✅"
    print(f"  {flag} {col:30s} | dtype pandas = {str(dtype):10s} | type Python = {type_valeur:10s} | ex: {exemple}")

🔍 Diagnostic des types de colonnes :

  ✅ Age                            | dtype pandas = int64      | type Python = int64      | ex: 60
  ✅ Experience                     | dtype pandas = int64      | type Python = int64      | ex: 36
  ✅ Revenu                         | dtype pandas = int64      | type Python = int64      | ex: 89000
  ✅ ZIP Code                       | dtype pandas = int64      | type Python = int64      | ex: 91745
  ✅ Family                         | dtype pandas = int64      | type Python = int64      | ex: 2
  ✅ Depenses_Mensuelle             | dtype pandas = float64    | type Python = float64    | ex: 2800.0
  ✅ Education                      | dtype pandas = str        | type Python = str        | ex: Licence
  ✅ Prêt_Hypothecaire              | dtype pandas = int64      | type Python = int64      | ex: 0
  ✅ Compte_Titre                   | dtype pandas = str        | type Python = str        | ex: Non
  ✅ Certificat_Depot               | dtype pandas = str  

In [23]:
# ==========================================
# 4. MODÈLE AVEC SMOTE
# ==========================================
print("\n" + "="*50)
print("MODÈLE AVEC SMOTE")
print("="*50)

# Utilisation de ImbPipeline pour que SMOTE ne s'applique QUE sur le train set
pipeline_smote = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

pipeline_smote.fit(X_train, y_train)
y_pred_smote = pipeline_smote.predict(X_test)

print("Rapport de classification (SMOTE) :")
print(classification_report(y_test, y_pred_smote))
print("Matrice de confusion (SMOTE) :")
print(confusion_matrix(y_test, y_pred_smote))




MODÈLE AVEC SMOTE


ValueError: scipy.sparse does not support dtype object. The only supported types are: bool, int8, uint8, int16, uint16, intc, uintc, int32, uint32, int64, uint64, float32, float64, longdouble, complex64, complex128, clongdouble.

In [ ]:
# ==========================================
# 5. AJUSTEMENT DU SEUIL (THRESHOLD) ET COURBE PR
# ==========================================
print("\n" + "="*50)
print("AJUSTEMENT DU SEUIL DE DÉCISION")
print("="*50)

# Obtenir les probabilités (classe 1) plutôt que les prédictions directes
y_probs = pipeline_smote.predict_proba(X_test)[:, 1]

# Calcul de la courbe Precision-Recall
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

# Trouver le seuil qui maximise le F1-Score
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10) # Ajout d'epsilon pour éviter division par 0
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Seuil optimal trouvé : {optimal_threshold:.3f}")
print(f"Meilleur F1-Score à ce seuil : {f1_scores[optimal_idx]:.3f}")

# Tracer la courbe Precision-Recall
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label='Courbe Precision-Recall')
plt.scatter(recall[optimal_idx], precision[optimal_idx], color='red', label=f'Seuil optimal = {optimal_threshold:.2f}')
plt.xlabel('Recall (Rappel)')
plt.ylabel('Precision')
plt.title('Courbe Precision-Recall avec Seuil Optimal')
plt.legend()
plt.grid(True)
plt.show()

# Appliquer le seuil personnalisé (ex: 0.3 ou le seuil optimal trouvé)
SEUIL_CHOISI = 0.30 # Vous pouvez remplacer par optimal_threshold
y_pred_custom = (y_probs >= SEUIL_CHOISI).astype(int)

print(f"\nRapport de classification avec seuil = {SEUIL_CHOISI} :")
print(classification_report(y_test, y_pred_custom))
print("Matrice de confusion (Seuil ajusté) :")
print(confusion_matrix(y_test, y_pred_custom))

In [ ]:
# ==========================================
# 6. SAUVEGARDE DU MODÈLE
# ==========================================
print("\n" + "="*50)
print("SAUVEGARDE DU MODÈLE")
print("="*50)

# On sauvegarde le pipeline complet (préprocesseur + SMOTE + modèle)
# Ainsi, si de nouvelles données arrivent, on pourra les passer directement dans le pipeline
nom_fichier_modele = 'modele_acceptation_pret.pkl'
joblib.dump(pipeline_smote, nom_fichier_modele)
print(f"Modèle sauvegardé avec succès sous : {nom_fichier_modele}")

# Exemple de rechargement du modèle (pour plus tard)
# modele_charge = joblib.load('modele_acceptation_pret.pkl')
# nouvelles_predictions = modele_charge.predict(nouvelles_donnees)